In [8]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [9]:
# ==========================================
# 1. CONFIGURATION & HYPERPARAMETERS
# ==========================================
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 5
DATASET_DIR = r"D:\Reports\PlantVillage"
# Update path to your extracted dataset directory)

In [10]:
# ==========================================
# 2. DATA PIPELINE (TRAIN/VAL SPLIT)
# ==========================================
print("[INFO] Loading and splitting dataset...")

train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_dataset.class_names
print(f"[INFO] Classes detected: {class_names}")

# Configure dataset performance with caching and prefetching
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)

[INFO] Loading and splitting dataset...
Found 20638 files belonging to 15 classes.
Using 16511 files for training.
Found 20638 files belonging to 15 classes.
Using 4127 files for validation.
[INFO] Classes detected: ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [11]:
# ==========================================
# 3. BUILD TRANSFER LEARNING MODEL (MobileNetV2)
# ==========================================
print("[INFO] Constructing MobileNetV2 Transfer Learning Architecture...")

# Load MobileNetV2 base pre-trained on ImageNet without top classification layers
base_model = MobileNetV2(input_shape=(224, 224, 3),
                         include_top=False,
                         weights='imagenet')

# Freeze the base model feature extractor
base_model.trainable = False

# Attach custom dense classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(len(class_names), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=outputs)

# Compile model
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

[INFO] Constructing MobileNetV2 Transfer Learning Architecture...


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,589,775 (9.88 MB)

 Trainable params: 331,791 (1.27 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [14]:
# ==========================================
# 4. MODEL TRAINING (FINE-TUNING PHASE)
# ==========================================
print("[INFO] Starting model training...")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS
)

# Save the trained weights
model.save("plant_disease_model.h5")
print("[INFO] Model training complete and saved as 'plant_disease_model.h5'")

[INFO] Starting model training...
Epoch 1/5
516/516 ━━━━━━━━━━━━━━━━━━━━ 368s 714ms/step - accuracy: 0.6657 - loss: 1.0177 - val_accuracy: 0.7495 - val_loss: 0.7461
Epoch 2/5
516/516 ━━━━━━━━━━━━━━━━━━━━ 344s 666ms/step - accuracy: 0.7389 - loss: 0.7880 - val_accuracy: 0.7739 - val_loss: 0.6652
Epoch 3/5
516/516 ━━━━━━━━━━━━━━━━━━━━ 368s 713ms/step - accuracy: 0.7656 - loss: 0.7030 - val_accuracy: 0.8093 - val_loss: 0.5667
Epoch 4/5
516/516 ━━━━━━━━━━━━━━━━━━━━ 340s 631ms/step - accuracy: 0.7891 - loss: 0.6224 - val_accuracy: 0.8163 - val_loss: 0.5473
Epoch 5/5
516/516 ━━━━━━━━━━━━━━━━━━━━ 367s 711ms/step - accuracy: 0.7998 - loss: 0.5845 - val_accuracy: 0.8340 - val_loss: 0.4912


[INFO] Model training complete and saved as 'plant_disease_model.h5'


In [17]:
# ==========================================
# 5. INFERENCE / TESTING ON A SINGLE INPUT IMAGE
# ==========================================
def predict_plant_disease(image_path):
    """
    Function to ingest an input leaf image path and output the predicted disease.
    """
    if not os.path.exists(image_path):
        print(f"[ERROR] Test image path not found: {image_path}")
        return

    # Load and resize image
    img = image.load_img(image_path, target_size=IMAGE_SIZE)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # Convert to batch dimension (1, 224, 224, 3)
    img_array = img_array / 255.0                 # Rescale pixel values if required

    # Perform prediction
    predictions = model.predict(img_array)
    predicted_index = np.argmax(predictions[0])
    confidence_score = float(np.max(predictions[0])) * 100

    predicted_label = class_names[predicted_index]

    print("\n================== INFERENCE RESULT ==================")
    print(f"📁 Input File: {image_path}")
    print(f"🌿 Predicted Class: {predicted_label}")
    print(f"📊 Confidence Score: {confidence_score:.2f}%")
    print("=======================================================")


In [18]:
# Example placeholder execution for single image input testing:
# Provide a valid image path from your test directory to get output
sample_test_image = r"D:\Reports\PlantVillage\Tomato_Early_blight\Sample.jpg"

# 2. Call the prediction function
predict_plant_disease(sample_test_image)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step

================== INFERENCE RESULT ==================
📁 Input File: D:\Reports\PlantVillage\Tomato_Early_blight\Sample.jpg
🌿 Predicted Class: Tomato_Late_blight
📊 Confidence Score: 100.00%
